In [ ]:
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

sns.set_theme(style="whitegrid")

conn = sqlite3.connect("dsl.db")

from dsl_statistics.db import RANK_NAMES

In [ ]:
# Get latest stats per player (most recent snapshot)
players_df = pd.read_sql("""
    SELECT
        p.id as player_id, p.display_name, p.steam_account_id,
        p.first_game_at, p.steam_account_created, p.steam_games_owned,
        p.steam_profile_visible,
        d.name as division, t.name as team,
        tm.role, tm.is_poc,
        ps.pp_score, ps.rank_number, ps.rank_subrank, ps.scraped_at
    FROM players p
    JOIN team_members tm ON p.id = tm.player_id AND tm.left_at IS NULL
    JOIN teams t ON tm.team_id = t.id
    JOIN divisions d ON t.division_id = d.id
    LEFT JOIN player_stats ps ON p.id = ps.player_id
        AND ps.scraped_at = (
            SELECT MAX(ps2.scraped_at) FROM player_stats ps2 WHERE ps2.player_id = p.id
        )
    ORDER BY d.name, t.name, p.display_name
""", conn)

players_df["rank_label"] = players_df["rank_number"].map(RANK_NAMES)
print(f"Loaded {len(players_df)} active players across {players_df['division'].nunique()} divisions")
players_df.head()

In [ ]:
div_stats = players_df.groupby("division")["pp_score"].agg(
    ["count", "mean", "median", "std", "min", "max"]
).round(1)
div_stats.columns = ["Players", "Avg PP", "Median PP", "Std Dev", "Min PP", "Max PP"]
print("=== Division Overview ===")
div_stats

In [ ]:
# Average PP of core players per team
core_players = players_df[players_df["role"] == "core"]
team_rankings = core_players.groupby(["division", "team"])["pp_score"].agg(
    ["mean", "count"]
).round(1)
team_rankings.columns = ["Avg Core PP", "Core Players"]
team_rankings = team_rankings.sort_values(["Avg Core PP"], ascending=False)

for div in sorted(players_df["division"].unique()):
    div_teams = team_rankings.loc[div].sort_values("Avg Core PP", ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(4, len(div_teams) * 0.4)))
    div_teams["Avg Core PP"].plot(
        kind="barh", ax=ax, color=sns.color_palette("viridis", len(div_teams))
    )
    ax.set_title(f"{div} — Team Rankings by Avg Core PP")
    ax.set_xlabel("Average PP (Core Players)")
    plt.tight_layout()
    plt.show()

In [ ]:
team_pp = core_players.groupby(["division", "team"])["pp_score"].mean()

print("=== Outlier Teams (>1 std dev from division mean) ===\n")
for div in sorted(players_df["division"].unique()):
    div_team_pp = team_pp[div]
    mean = div_team_pp.mean()
    std = div_team_pp.std()
    outliers = div_team_pp[(div_team_pp > mean + std) | (div_team_pp < mean - std)]
    if len(outliers) > 0:
        print(f"{div} (mean={mean:.1f}, std={std:.1f}):")
        for team_name, pp in outliers.items():
            direction = "ABOVE" if pp > mean else "BELOW"
            print(f"  {team_name}: {pp:.1f} ({direction}, {abs(pp - mean)/std:.1f}σ)")
        print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PP histogram per division
for div in sorted(players_df["division"].unique()):
    div_data = players_df[players_df["division"] == div]["pp_score"].dropna()
    axes[0].hist(div_data, bins=20, alpha=0.5, label=div)
axes[0].set_title("PP Distribution by Division")
axes[0].set_xlabel("PP Score")
axes[0].set_ylabel("Count")
axes[0].legend()

# Rank distribution
rank_counts = players_df["rank_label"].value_counts()
rank_order = [RANK_NAMES[i] for i in range(12) if RANK_NAMES[i] in rank_counts.index]
rank_counts = rank_counts.reindex(rank_order)
rank_counts.plot(
    kind="bar", ax=axes[1], color=sns.color_palette("coolwarm", len(rank_counts))
)
axes[1].set_title("Rank Distribution (All Divisions)")
axes[1].set_xlabel("Rank")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
def scout_player(player_name: str):
    """Display scouting info for a player."""
    player = players_df[
        players_df["display_name"].str.contains(player_name, case=False)
    ]
    if player.empty:
        print(f"Player '{player_name}' not found")
        return

    p = player.iloc[0]
    print(f"=== {p['display_name']} ===")
    print(f"Team: {p['team']} ({p['division']})")
    print(f"Role: {p['role']}{'  [POC]' if p['is_poc'] else ''}")
    print(f"PP: {p['pp_score']:.1f}")
    print(f"Rank: {p['rank_label']} {p['rank_subrank']}")
    print(f"Steam Account Age: {p['steam_account_created'] or 'Unknown'}")
    print(f"Games Owned: {p['steam_games_owned'] or 'Unknown'}")
    print()

    # Top heroes
    heroes = pd.read_sql("""
        SELECT ph.hero_name, ph.matches_played, ph.win_rate
        FROM player_heroes ph
        JOIN player_stats ps ON ph.stats_id = ps.id
        WHERE ps.player_id = ?
        AND ps.scraped_at = (SELECT MAX(scraped_at) FROM player_stats WHERE player_id = ?)
        ORDER BY ph.matches_played DESC
        LIMIT 10
    """, conn, params=(int(p["player_id"]), int(p["player_id"])))

    if not heroes.empty:
        print("Top Heroes:")
        for _, h in heroes.iterrows():
            print(
                f"  {h['hero_name']}: {h['matches_played']} games, "
                f"{h['win_rate']*100:.0f}% WR"
            )

    # PP trend from match history
    matches = pd.read_sql("""
        SELECT match_date, pp_after, pp_change, hero_name, result
        FROM player_matches
        WHERE player_id = ?
        ORDER BY match_date DESC
        LIMIT 20
    """, conn, params=(int(p["player_id"]),))

    if not matches.empty and matches["pp_after"].notna().any():
        fig, ax = plt.subplots(figsize=(10, 4))
        matches_sorted = matches.sort_values("match_date")
        colors = [
            "green" if r == "win" else "red" for r in matches_sorted["result"]
        ]
        ax.plot(
            range(len(matches_sorted)),
            matches_sorted["pp_after"],
            marker="o",
            color="steelblue",
        )
        ax.scatter(
            range(len(matches_sorted)),
            matches_sorted["pp_after"],
            c=colors,
            zorder=5,
        )
        ax.set_title(f"{p['display_name']} — Recent PP Trend")
        ax.set_xlabel("Match")
        ax.set_ylabel("PP")
        plt.tight_layout()
        plt.show()


# Example usage:
# scout_player("YourName")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
div_order = sorted(players_df["division"].unique())
data = [
    players_df[players_df["division"] == d]["pp_score"].dropna() for d in div_order
]
ax.boxplot(data, labels=div_order)
ax.set_title("PP Distribution Across Divisions")
ax.set_ylabel("PP Score")
plt.tight_layout()
plt.show()

In [ ]:
# PP trajectory from match history for top players in a division
my_division = "Division 2"  # Change as needed

div_players = players_df[players_df["division"] == my_division].nlargest(
    5, "pp_score"
)

fig, ax = plt.subplots(figsize=(12, 5))
for _, p in div_players.iterrows():
    matches = pd.read_sql("""
        SELECT match_date, pp_after FROM player_matches
        WHERE player_id = ? AND pp_after IS NOT NULL
        ORDER BY match_date
    """, conn, params=(int(p["player_id"]),))
    if not matches.empty:
        ax.plot(
            range(len(matches)),
            matches["pp_after"],
            label=p["display_name"],
            marker=".",
        )

ax.set_title(f"PP Trends — Top 5 Players in {my_division}")
ax.set_xlabel("Match Index")
ax.set_ylabel("PP")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
alt_flags = players_df[
    (players_df["pp_score"].notna())
    & (
        (players_df["steam_profile_visible"] == False)
        | (
            players_df["steam_games_owned"].notna()
            & (players_df["steam_games_owned"] < 10)
        )
        | (
            players_df["steam_account_created"].notna()
            & (
                pd.to_datetime(players_df["steam_account_created"], utc=True)
                > pd.Timestamp("2024-06-01", tz="UTC")
            )
        )
    )
].sort_values("pp_score", ascending=False)

print(f"=== Potential Alt Accounts ({len(alt_flags)} flagged) ===\n")
for _, p in alt_flags.iterrows():
    flags = []
    if p["steam_profile_visible"] == False:
        flags.append("PRIVATE PROFILE")
    if pd.notna(p["steam_games_owned"]) and p["steam_games_owned"] < 10:
        flags.append(f"ONLY {int(p['steam_games_owned'])} GAMES")
    if pd.notna(p["steam_account_created"]):
        created = pd.to_datetime(p["steam_account_created"], utc=True)
        if created > pd.Timestamp("2024-06-01", tz="UTC"):
            flags.append(f"ACCOUNT CREATED {created.strftime('%Y-%m')}")

    print(f"  {p['display_name']} ({p['team']}, {p['division']})")
    print(
        f"    PP: {p['pp_score']:.1f} | Rank: {p['rank_label']} | "
        f"Flags: {', '.join(flags)}"
    )

In [ ]:
conn.close()